# BreezyVoice on Google Colab

Experiment with [mtkresearch/BreezyVoice](https://github.com/mtkresearch/BreezyVoice) — MediaTek Research's voice-cloning TTS for **Taiwanese Mandarin** (adapted from CosyVoice, paper: [arXiv:2501.17790](https://arxiv.org/abs/2501.17790)).

## Before you run

- **GPU runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU**. A T4 is sufficient.
- **Python 3.10 is hard-required**. BreezyVoice depends on `ttsfrd-0.3.9-cp310-cp310-linux_x86_64.whl`, which is Python-3.10-only. Modern Colab ships Python 3.11, so this notebook uses `uv` to bootstrap a clean 3.10 venv and runs everything through it.
- **Install is slow** (~10–20 min): the upstream `requirements.txt` pulls `torch==2.3.1+cu118`, `onnxruntime-gpu`, `deepspeed`, `lightning`, and the two ttsfrd wheels from ModelScope. We set `DS_BUILD_OPS=0` so DeepSpeed skips ahead-of-time op builds (it JIT-compiles at runtime instead) — otherwise the install can hang for ten minutes compiling ops we may never use.
- Model weights (`MediaTek-Research/BreezyVoice` on Hugging Face) download on first inference.

## What this notebook does

1. Sanity-check runtime + GPU.
2. Install `uv`, create a Python 3.10 venv at `/content/BreezyVoice/.venv`.
3. Clone the repo and `uv pip install -r requirements.txt` into the venv.
4. Run `single_inference.py` against the bundled `data/example.wav` speaker prompt.
5. Play the result inline.
6. Re-run with your own reference audio + your own text.

## 1. Sanity-check the runtime

In [ ]:
!python3 --version
!nvidia-smi | head -n 20

## 2. Install `uv` and a Python 3.10 toolchain

`uv` will download CPython 3.10 for us if it isn't installed.

In [ ]:
!pip install -q uv
!uv python install 3.10

## 3. Clone BreezyVoice and create the venv

In [ ]:
%cd /content
![ -d BreezyVoice ] || git clone https://github.com/mtkresearch/BreezyVoice.git
%cd /content/BreezyVoice
!uv venv --python 3.10 .venv
!.venv/bin/python --version

## 4. Install requirements into the venv

This is the slow step. `DS_BUILD_OPS=0` is set so DeepSpeed installs without compiling its ops up front.

In [ ]:
%env DS_BUILD_OPS=0
%env PYTHONUTF8=1
!cd /content/BreezyVoice && uv pip install --python .venv/bin/python -r requirements.txt

## 5. Verify the venv sees the GPU

Confirms the venv's torch picks up CUDA. If `cuda.is_available()` is False here, every inference below will silently fall back to CPU and be extremely slow.

In [ ]:
!cd /content/BreezyVoice && .venv/bin/python -c "import torch; print('torch:', torch.__version__); print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')"

## 6. Run the bundled single-inference example

Uses `data/example.wav` as the speaker prompt and synthesizes "歡迎使用聯發創新基地 BreezyVoice 模型。" — matches `run_single_inference.sh` upstream. Output lands at `results/out.wav`.

First run also downloads the model from Hugging Face (~a few GB).

In [ ]:
%cd /content/BreezyVoice
!mkdir -p results
!PYTHONUTF8=1 .venv/bin/python single_inference.py \
    --speaker_prompt_audio_path "data/example.wav" \
    --speaker_prompt_text_transcription "在密碼學中，加密是將明文資訊改變為難以讀取的密文內容，使之不可讀的方法。只有擁有解密方法的對象，經由解密過程，才能將密文還原為正常可讀的內容。" \
    --content_to_synthesize "歡迎使用聯發創新基地 BreezyVoice 模型。" \
    --output_path results/out.wav

## 7. Play the result inline

In [ ]:
from IPython.display import Audio, display
print('--- speaker reference (data/example.wav) ---')
display(Audio('/content/BreezyVoice/data/example.wav'))
print('--- synthesized output (results/out.wav) ---')
display(Audio('/content/BreezyVoice/results/out.wav'))

## 8. Clone your own voice

Upload a short clean recording of the voice you want to clone (10–30 seconds is plenty). Mono WAV is easiest; the system also accepts what soundfile can read.

**Tip**: providing the transcription via `--speaker_prompt_text_transcription` is *highly recommended* — without it the script falls back to Whisper, which adds a few seconds and can mis-hear domain terms.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    print('uploaded:', name)

In [ ]:
# Edit these three values to match your uploaded file and target text.
SPEAKER_AUDIO = '/content/your_voice.wav'           # path to the file you just uploaded
SPEAKER_TEXT  = ''                                   # transcription of SPEAKER_AUDIO; '' to auto-transcribe via Whisper
TARGET_TEXT   = '今天天氣真好，要不要去散步？'

import shlex, subprocess, os
cmd = [
    '.venv/bin/python', 'single_inference.py',
    '--speaker_prompt_audio_path', SPEAKER_AUDIO,
    '--content_to_synthesize',     TARGET_TEXT,
    '--output_path',               'results/custom.wav',
]
if SPEAKER_TEXT:
    cmd += ['--speaker_prompt_text_transcription', SPEAKER_TEXT]

env = {**os.environ, 'PYTHONUTF8': '1'}
print('$', ' '.join(shlex.quote(c) for c in cmd))
subprocess.run(cmd, cwd='/content/BreezyVoice', env=env, check=True)

In [ ]:
from IPython.display import Audio, display
display(Audio('/content/BreezyVoice/results/custom.wav'))

## Notes

- **Phonetic control**: BreezyVoice supports manual 注音 (bopomofo) correction inline in the target text, e.g. `"今天天氣真好[:ㄏㄠ3]"`. Use sparingly — the auto-annotator handles most cases.
- **Batch inference**: see `batch_inference.py` and `data/batch_files.csv` upstream — run via `.venv/bin/python batch_inference.py --csv_file ... --speaker_prompt_audio_folder ... --output_audio_folder ...`.
- **Gradio playground**: the upstream repo also has a Docker + OpenAI-compatible API path (`docker compose up`). That doesn't translate cleanly to Colab; the [HF Spaces playground](https://huggingface.co/spaces/Splend1dchan/BreezyVoice-Playground) is the easier route for a UI.
- **Why `uv`?** It's the cleanest way to get Python 3.10 in a Colab cell. `condacolab` works too but requires a kernel restart and adds a 2-minute miniconda install on top.